# XAI-Compress Research Notebook

This notebook documents dataset statistics, model architecture, training metrics, and honest codec comparisons.
It does **not** claim superiority over 7-Zip unless a measured table shows it.

## 1–4. Dataset analysis, statistics, distribution, entropy

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from xai_compress.datasets.corpus import write_corpus
from xai_compress.analyzer import analyze_block, classify_kind, byte_entropy

root = write_corpus(Path('data/notebook_corpus'))
rows = []
for p in sorted(root.rglob('*')):
    if p.is_file():
        data = p.read_bytes()
        a = analyze_block(data)
        rows.append((p.as_posix(), classify_kind(data), len(data), a.entropy, a.suggested, a.zlib_size))
print('path | kind | size | entropy | strategy | zlib')
for r in rows:
    print(r)
ents = [r[3] for r in rows]
plt.figure(figsize=(8,3))
plt.bar(range(len(ents)), ents)
plt.ylabel('entropy (bits/byte)')
plt.title('Per-file order-0 entropy')
plt.show()

## 5. Model architecture

- **GRU v1**: streaming recurrent byte model, constant state, best for long files.
- **Transformer v1**: causal self-attention over a bounded context, KV-cache on decode.
- Both emit 256 logits per byte; integer frequencies drive arithmetic coding or rANS.
- Hybrid path can store, zlib, static AC, or neural per chunk after entropy analysis.

## 6–10. Training, loss, validation, BPB, ratio

In [ ]:
import csv
from pathlib import Path
p = Path('experiments/runs/tiny_gru.metrics.csv')
if p.exists():
    rows = list(csv.DictReader(p.open()))
    print(rows[-1] if rows else 'empty')
else:
    print('Train first: python scripts/run_experiments.py')

## 11–20. Speed, GPU, model comparison, classical codecs, errors, conclusions

In [ ]:
import json
from pathlib import Path
s = Path('experiments/runs/benchmark.summary.json')
if s.exists():
    data = json.loads(s.read_text())
    print(f"{'codec':12} {'ratio':>8} {'bpb':>8} {'cMB/s':>8} {'dMB/s':>8}")
    for name, row in sorted(data.items(), key=lambda kv: kv[1]['compressed']):
        print(f"{name:12} {row['ratio']:8.3f} {row['bpb']:8.3f} {row['compress_mbs']:8.2f} {row['decompress_mbs']:8.2f}")
    print('\nScientific note: smaller compressed size wins. High-entropy files should show ratio ~1.0 for every codec.')
else:
    print('No benchmark summary yet.')